In [2]:
#his block initializes the necessary environment and starts a Spark session, crucial for loading, transforming, and analyzing data.
import ConnectionConfig as cc
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import datetime
import requests
from delta import DeltaTable

# Set up the environment and start Spark session
cc.setupEnvironment()
spark = cc.startLocalCluster("DIM_WEATHER", 4)
spark.getActiveSession()


In [3]:
# Read CSV file with weather data
df_weather = spark.read.csv("./FileStore/tables/weather_data.csv", header=True, inferSchema=True)
df_weather.show()

+----------+------------+---------+------------+
|weather_id| temperature|condition|weather_type|
+----------+------------+---------+------------+
|         1|        >=15|    Sunny|   aangenaam|
|         2|         <15|    Rainy| onaangenaam|
|         3|<15 and >-10|   Cloudy|    neutraal|
|         4|         any|  unknown|    onbekend|
+----------+------------+---------+------------+



In [4]:
#Transform Data:
# Transform columns if needed, for example, renaming for clarity
dimWeather = df_weather.withColumnRenamed("weather_id", "weatherSK") \
    .withColumnRenamed("temperature", "temperature_condition") \
    .withColumnRenamed("condition", "weather_condition") \
    .withColumnRenamed("weather_type", "weather_category")

# Define path for the Delta table
weather_delta_path = "./spark-warehouse/weather_data_delta"  
# Save dimWeather DataFrame as a Delta table
dimWeather.write.format("delta").mode("overwrite").save(weather_delta_path)

print("dimWeather table has been saved successfully to ./spark/weather_data_delta")

# Show transformed DataFrame
dimWeather.show()


dimWeather table has been saved successfully to ./spark/weather_data_delta
+---------+---------------------+-----------------+----------------+
|weatherSK|temperature_condition|weather_condition|weather_category|
+---------+---------------------+-----------------+----------------+
|        1|                 >=15|            Sunny|       aangenaam|
|        2|                  <15|            Rainy|     onaangenaam|
|        3|         <15 and >-10|           Cloudy|        neutraal|
|        4|                  any|          unknown|        onbekend|
+---------+---------------------+-----------------+----------------+



In [5]:
# Create or replace the temporary view for weather dimension
dimWeather.createOrReplaceTempView("dimWeather")

# Display the dimension table
spark.sql("SELECT * FROM dimWeather").show()


+---------+---------------------+-----------------+----------------+
|weatherSK|temperature_condition|weather_condition|weather_category|
+---------+---------------------+-----------------+----------------+
|        1|                 >=15|            Sunny|       aangenaam|
|        2|                  <15|            Rainy|     onaangenaam|
|        3|         <15 and >-10|           Cloudy|        neutraal|
|        4|                  any|          unknown|        onbekend|
+---------+---------------------+-----------------+----------------+



In [6]:
# Write DataFrame to Delta table
dimWeather.write.format("delta").mode("overwrite").saveAsTable("dimWeather")


In [7]:
spark.stop()